# 2) Other ways to solve the power flow — Exercises

**Goals:** write the Gauss–Seidel update and test it at the solution, check what the FDLF matrix is made of, tune Gauss–Seidel on a larger network, compare the two FDLF variants, and measure what reusing a factorisation is worth over a time series.

The setup cell holds the lecture's `network(name, r_scale)` loader, `mismatch` and the sparse `newton_sparse` (which also accepts other injections `S` and a start `V0`). Everything is in per unit.

**Submit** your notebook for review before opening the solution in `solutions/`.

In [ ]:
import logging
import time
import warnings

import numpy as np
import pandas as pd
import pandapower as pp
import pandapower.networks as pn
import scipy.sparse as sp
import scipy.sparse.linalg as spl
from pandapower.pypower.makeB import makeB

warnings.filterwarnings("ignore")
logging.getLogger("pandapower").setLevel(logging.ERROR)


def network(name, r_scale=1.0):
    """The lecture's loader: a solved pandapower case and its internal data, slack angle included."""
    net = getattr(pn, name)()
    net.line.r_ohm_per_km *= r_scale
    pp.runpp(net, init="flat", numba=False, tolerance_mva=1e-10)
    internal, order = net._ppc["internal"], net._pd2ppc_lookups["bus"]
    V0 = np.ones(len(internal["Sbus"]), dtype=complex)
    V0[order[net.gen.bus.to_numpy()]] = net.gen.vm_pu.to_numpy()
    V0[order[net.ext_grid.bus.to_numpy()]] = (net.ext_grid.vm_pu * np.exp(1j * np.radians(net.ext_grid.va_degree))).to_numpy()
    return {"net": net, "Y": internal["Ybus"].tocsr(), "S": internal["Sbus"].copy(), "V0": V0, "pv": internal["pv"],
            "pq": internal["pq"], "bus": internal["bus"], "branch": internal["branch"], "base": net._ppc["baseMVA"],
            "order": order, "V_pandapower": internal["V"]}


def mismatch(d, V, S=None):
    S = d["S"] if S is None else S
    m = V * np.conj(d["Y"] @ V) - S
    return max(np.abs(m[np.r_[d["pv"], d["pq"]]].real).max(), np.abs(m[d["pq"]].imag).max())


def newton_sparse(d, S=None, V0=None, tol=1e-8, max_iter=50):
    """The lecture's sparse Newton-Raphson (optionally with other injections S and start V0)."""
    Y, S = d["Y"], d["S"] if S is None else S
    V = (d["V0"] if V0 is None else V0).copy()
    pv, pq = d["pv"], d["pq"]
    pvpq = np.r_[pv, pq]
    for it in range(max_iter):
        I = Y @ V
        m = V * np.conj(I) - S
        F = np.r_[m[pvpq].real, m[pq].imag]
        if np.abs(F).max() < tol:
            return V, it
        dV, dI, dVn = sp.diags(V), sp.diags(I), sp.diags(V / np.abs(V))
        dS_dVa = 1j * dV @ np.conj(dI - Y @ dV)
        dS_dVm = dV @ np.conj(Y @ dVn) + np.conj(dI) @ dVn
        J = sp.vstack([sp.hstack([dS_dVa[pvpq][:, pvpq].real, dS_dVm[pvpq][:, pq].real]),
                       sp.hstack([dS_dVa[pq][:, pvpq].imag, dS_dVm[pq][:, pq].imag])]).tocsc()
        dx = spl.spsolve(J, -F)
        Va, Vm = np.angle(V), np.abs(V)
        Va[pvpq] += dx[:len(pvpq)]
        Vm[pq] += dx[len(pvpq):]
        V = Vm * np.exp(1j * Va)
    raise RuntimeError("Newton: no convergence")

### Warm-ups

**1. The Gauss–Seidel update**

Write `gs_update(Y, V, S_i, i, is_pv, v_set)`: the new voltage of bus i from the lecture's formula (without acceleration) — for a PV bus, compute its reactive injection from the current voltages first and rescale the result to `v_set`. Test the defining property: at pandapower's solution, the update must return the bus's voltage unchanged, for every PV and PQ bus of the 14-bus system.

In [ ]:
def gs_update(Y: np.ndarray, V: np.ndarray, S_i: complex, i: int, is_pv: bool, v_set: float) -> complex:
    """One Gauss-Seidel update of bus i."""
    ...  # YOUR CODE HERE

In [ ]:
d14 = network("case14")
Y14, V_star = d14["Y"].toarray(), d14["V_pandapower"]
pv_set = set(d14["pv"].tolist())
changes = [abs(gs_update(Y14, V_star, d14["S"][i], i, i in pv_set, abs(V_star[i])) - V_star[i])
           for i in np.r_[d14["pv"], d14["pq"]]]
assert max(changes) < 1e-8, "the solution is a fixed point of the update"
print(f"largest change at the solution: {max(changes):.1e} p.u.")

**2. What is B′ made of?**

Build B′ and B″ with `makeB(base, bus, branch, 2)` for the 14-bus system. Check that B′ is symmetric, and that for every line without an off-nominal tap the off-diagonal entry B′[i, k] equals −1/x of the line (x in p.u., from column 3 of `branch`; parallel lines add). Write `check_bprime(d)` returning the largest deviation.

In [ ]:
def check_bprime(d) -> float:
    """Largest deviation between B'[i, k] and the sum of -1/x over the branches joining i and k (no taps)."""
    ...  # YOUR CODE HERE

In [ ]:
deviation = check_bprime(d14)
assert deviation < 1e-9
print(f"B' off-diagonals equal -1/x to {deviation:.1e}")

### Core

**3. Gauss–Seidel on a larger network**

Implement `gauss_seidel(d, alpha, tol=1e-8, max_sweeps=20000)` with your `gs_update` (and the acceleration factor applied before the PV rescaling, as in the lecture). Run it on the **118-bus** system for α in [1.0, 1.4, 1.6, 1.8] and report the sweeps needed. Compare with the 14-bus results of the lecture (245 sweeps at α = 1.0, 51 at α = 1.6).

In [ ]:
def gauss_seidel(d, alpha: float = 1.0, tol: float = 1e-8, max_sweeps: int = 20_000):
    """Gauss-Seidel power flow with acceleration. Returns V and the number of sweeps."""
    ...  # YOUR CODE HERE

In [ ]:
d118 = network("case118")
rows = {}
for alpha in [1.0, 1.4, 1.6, 1.8]:
    V, sweeps = gauss_seidel(d118, alpha)
    assert np.abs(V - d118["V_pandapower"]).max() < 1e-6
    rows[alpha] = sweeps
gs118 = pd.Series(rows, name="sweeps, 118 buses").rename_axis("alpha")
assert gs118.min() > 5 * 51, "far more sweeps than on 14 buses"
gs118

**4. Two FDLF variants**

`makeB` builds B′ and B″ in two standard variants: **XB** (`alg=2`, the lecture's: resistance ignored in B′) and **BX** (`alg=3`: resistance ignored in B″ instead). Write `fdlf(d, variant)` and compare the iterations of both on the 14-bus system with line resistance scaled ×1, ×2, ×3 and ×4.

In [ ]:
def fdlf(d, variant: int = 2, S=None, V0=None, tol=1e-8, max_iter=500, factors=None):
    """Fast decoupled load flow (variant 2 = XB, 3 = BX). Returns V, iterations and the factorisations."""
    ...  # YOUR CODE HERE

In [ ]:
rows = {}
for r_scale in [1, 2, 3, 4]:
    d = network("case14", r_scale)
    rows[r_scale] = {"XB": fdlf(d, 2)[1], "BX": fdlf(d, 3)[1], "Newton": newton_sparse(d)[1]}
variants = pd.DataFrame(rows).T.rename_axis("line resistance x")
assert variants.loc[1, "XB"] == 8, "the lecture's 8 iterations"
variants

**5. Interpretation**

How does Gauss–Seidel's sweep count scale from 14 to 118 buses, and does the best acceleration factor carry over? What did you find about the two FDLF variants as resistance grows — and does either one remove the weakness?

*Your answer here.*

### Challenge

**6. A time series: factorise once, solve many**

Solve **200 load snapshots** of the 1,354-bus system (`case1354pegase`): in snapshot t every load is scaled by a factor drawn uniformly from 0.9–1.1 (seed 0), generators' active power unchanged — so S_t = S_gen − factor_t · S_load. Compare two strategies: (a) `newton_sparse` for each snapshot, starting from the previous snapshot's solution; (b) `fdlf` with B′ and B″ **factorised once** and reused for all 200 snapshots, also warm-started. Report total time, total iterations and the largest difference between the two strategies' voltages. Write `time_series(n=200)`. (Not every test case suits this: the 300-bus case, with its generators fixed, has no solution already at 0.9 or 1.05 times its base load — pandapower's solvers fail there too.)

In [ ]:
def time_series(n: int = 200) -> dict:
    """Total time (s) and iterations of Newton and of FDLF with one factorisation over n load snapshots."""
    ...  # YOUR CODE HERE

In [ ]:
ts = time_series()
assert ts["largest_difference"] < 1e-6, "both strategies solve the same 200 power flows"
summary = pd.DataFrame({k: {"seconds": v["seconds"], "iterations": v["iterations"]} for k, v in ts.items() if isinstance(v, dict)}).T
print(f"largest difference between the strategies: {ts['largest_difference']:.1e} p.u. (timings depend on the machine)")
summary.round(2)

**7. Interpretation**

How much faster is FDLF over the time series, and why is the advantage larger than in the lecture's single power flows? Name a real study where this pattern matters.

*Your answer here.*